# Stage Silver — INE Defunciones por Geografía

Transforma las 2 tablas INE de defunciones geográficas hacia `stage_silver_ss2`.

| Tabla Bronze | Tabla Silver | Descripción |
|---|---|---|
| `ine_defunciones_sexo_depto_residencia_causas_muerte` | `ine_defunciones_depto_residencia` | Defunciones por departamento de residencia |
| `ine_defunciones_causas_externas_sexo_depto_ocurrencia` | `ine_defunciones_causas_externas` | Causas externas por departamento de ocurrencia |

**Diferencia clave entre las dos tablas:**
- `depto_residencia`: columna depto = `departamento_de_residencia`, código CIE = `codigo_cie_10`
- `causas_externas`: columna depto = `departamento_de_ocurrencia`, campo `causas_externas_codigo_cie_10` combina descripción + rango en un solo campo: `"Accidentes de transporte (V01-V99)"` — Silver lo separa en `causa_de_muerte` + `cie_10_norm`

**Transformaciones aplicadas:**
- `anio`, `total`, `hombres`, `mujeres` → INT
- `cie_10_norm`: elimina separadores `: ` de códigos simples; `'Otras causas'` → `'R99'`; para `causas_externas` extrae el rango entre paréntesis (ej. `V01-V99`)
- `causa_de_muerte` (solo `causas_externas`): texto descriptivo extraído del campo combinado Bronze
- Filas con `departamento == 'Todos los departamentos'` **se filtran** (subtotal geográfico)
- Filas con `codigo_cie_10 == 'Todas las causas'` **se filtran** (subtotal de causa, solo en tabla residencia)
- Validación de cuadratura (`total == hombres + mujeres`) — se loguea como warning si falla, sin columna en tabla
- `departamento_oficial` → JOIN con catálogo ref (nombre normalizado con tildes y casing correcto)
- `dropDuplicates()`
- Auditoría Silver: `silver_processed_timestamp`, `silver_job_run_id`

In [ ]:
# ── Setup: imports, config, catálogo y helpers en una sola celda ─────────────
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

BRONZE_SCHEMA = "sandbox_bronze_ss2"
SILVER_SCHEMA = "stage_silver_ss2"
WRITE_MODE    = "overwrite"

# ── Catálogo de referencia ────────────────────────────────────────────────────
_DEPTO_ROWS = [
    ("ALTA VERAPAZ",            "Alta Verapaz"          ),
    ("BAJA VERAPAZ",            "Baja Verapaz"          ),
    ("CHIMALTENANGO",           "Chimaltenango"         ),
    ("CHIQUIMULA",              "Chiquimula"            ),
    ("EL PROGRESO",             "El Progreso"           ),
    ("ESCUINTLA",               "Escuintla"             ),
    ("GUATEMALA",               "Guatemala"             ),
    ("HUEHUETENANGO",           "Huehuetenango"         ),
    ("IZABAL",                  "Izabal"                ),
    ("JALAPA",                  "Jalapa"                ),
    ("JUTIAPA",                 "Jutiapa"               ),
    ("PETEN",                   "Petén"                 ),
    ("EL PETEN",                "Petén"                 ),
    ("EL PETÉN",                "Petén"                 ),
    ("PETÉN",                   "Petén"                 ),
    ("QUETZALTENANGO",          "Quetzaltenango"        ),
    ("QUICHE",                  "El Quiché"             ),
    ("QUICHÉ",                  "El Quiché"             ),
    ("EL QUICHE",               "El Quiché"             ),
    ("EL QUICHÉ",               "El Quiché"             ),
    ("RETALHULEU",              "Retalhuleu"            ),
    ("SACATEPEQUEZ",            "Sacatepéquez"          ),
    ("SACATEPÉQUEZ",            "Sacatepéquez"          ),
    ("SAN MARCOS",              "San Marcos"            ),
    ("SANTA ROSA",              "Santa Rosa"            ),
    ("SOLOLA",                  "Sololá"                ),
    ("SOLOLÁ",                  "Sololá"                ),
    ("SUCHITEPEQUEZ",           "Suchitepéquez"         ),
    ("SUCHITEPÉQUEZ",           "Suchitepéquez"         ),
    ("TOTONICAPAN",             "Totonicapán"           ),
    ("TOTONICAPÁN",             "Totonicapán"           ),
    ("ZACAPA",                  "Zacapa"                ),
]

ref_depto_df = spark.createDataFrame(
    _DEPTO_ROWS,
    ["depto_variante", "depto_oficial"]
)

# ── Helpers ───────────────────────────────────────────────────────────────────

def get_job_run_id():
    try:
        return (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext()
            .currentRunId().toString()
        )
    except Exception:
        return "manual"


def normalize_cie10(col_expr):
    """'Otras causas' → 'R99'  |  J:18:0 → J180  |  B:37 → B37  |  rangos como 'R00-R99' quedan igual"""
    return (
        F.when(col_expr.isin("Todas las causas", "Todas las causas externas"), F.lit("A00-Y98"))
         .when(col_expr == "Otras causas", F.lit("R99"))
         .otherwise(F.regexp_replace(col_expr, r'[: ]', ''))
    )


def join_departamento(df, depto_col, ref_df):
    return (
        df
        .join(
            F.broadcast(ref_df),
            F.upper(F.trim(F.col(depto_col))) == F.col("depto_variante"),
            "left",
        )
        .drop("depto_variante")
        .withColumnRenamed("depto_oficial", "departamento_oficial")
    )


def validate_cuadratura(df, table_name):
    """Loguea filas donde total ≠ hombres + mujeres. No filtra — solo alerta."""
    fail = df.limit(5000).filter(F.col("total") != (F.col("hombres") + F.col("mujeres"))).count()
    if fail > 0:
        logger.warning(f"[{table_name}] {fail} filas (muestra 5k) con total ≠ H+M — revisar fuente")
    else:
        logger.info(f"[{table_name}] Cuadratura OK en muestra 5k")


def add_silver_audit(df, run_id):
    return (
        df
        .withColumn("silver_processed_timestamp", F.current_timestamp())
        .withColumn("silver_job_run_id",           F.lit(run_id))
    )


RUN_ID = get_job_run_id()
logger.info(f"Setup OK — run_id={RUN_ID}")

## Tabla 1: ine_defunciones_sexo_depto_residencia_causas_muerte → ine_defunciones_depto_residencia

Esquema Bronze: `departamento_de_residencia, codigo_cie_10, causa_de_muerte, total, hombres, mujeres, anio`

Subtotales:
- `departamento_de_residencia = 'Todos los departamentos'` — fila nacional agregada
- `codigo_cie_10 = 'Todas las causas'` — fila de suma por departamento

In [ ]:
BRONZE_TABLE = "ine_defunciones_sexo_depto_residencia_causas_muerte"
SILVER_TABLE = "ine_defunciones_depto_residencia"

df = spark.table(f"{BRONZE_SCHEMA}.{BRONZE_TABLE}")
logger.info(f"[{BRONZE_TABLE}] Bronze: {df.count():,} filas")

df_silver = (
    df
    .withColumn("anio",        F.col("anio").cast(IntegerType()))
    .withColumn("total",       F.col("total").cast(IntegerType()))
    .withColumn("hombres",     F.col("hombres").cast(IntegerType()))
    .withColumn("mujeres",     F.col("mujeres").cast(IntegerType()))
    .withColumn("cie_10_norm", normalize_cie10(F.col("codigo_cie_10")))
    # Filtrar subtotales geográficos y de causa — Silver solo tiene filas granulares
    .filter(F.upper(F.trim(F.col("departamento_de_residencia"))) != F.lit("TODOS LOS DEPARTAMENTOS"))
    .filter(F.col("codigo_cie_10") != F.lit("Todas las causas"))
    .dropDuplicates()
)

validate_cuadratura(df_silver, SILVER_TABLE)
df_silver = join_departamento(df_silver, "departamento_de_residencia", ref_depto_df)
df_silver = add_silver_audit(df_silver, RUN_ID)

logger.info(f"[{SILVER_TABLE}] Silver: {df_silver.count():,} filas")
(
    df_silver.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_SCHEMA}.{SILVER_TABLE}")
)
logger.info(f"Escrito → {SILVER_SCHEMA}.{SILVER_TABLE}")

## Tabla 2: ine_defunciones_causas_externas_sexo_depto_ocurrencia → ine_defunciones_causas_externas

Esquema Bronze: `departamento_de_ocurrencia, causas_externas_codigo_cie_10, total, hombres, mujeres, anio`

**Diferencias respecto a la tabla de residencia:**
- Campo de departamento: `departamento_de_ocurrencia`
- `causas_externas_codigo_cie_10` combina descripción y rango CIE en un solo campo:
  `"Accidentes de transporte (V01-V99)"` → Silver descompone en:
  - `causa_de_muerte` = texto descriptivo antes del paréntesis (`"Accidentes de transporte"`)
  - `cie_10_norm` = rango entre paréntesis (`"V01-V99"`)
  - El campo Bronze original `causas_externas_codigo_cie_10` se conserva para trazabilidad
- Subtotal solo por departamento (no hay fila 'Todas las causas' en esta tabla)

In [ ]:
BRONZE_TABLE = "ine_defunciones_causas_externas_sexo_depto_ocurrencia"
SILVER_TABLE = "ine_defunciones_causas_externas"

df = spark.table(f"{BRONZE_SCHEMA}.{BRONZE_TABLE}")
logger.info(f"[{BRONZE_TABLE}] Bronze: {df.count():,} filas")

df_silver = (
    df
    # Eliminar filas sin código de causa externa — la columna no aporta si es NULL
    .filter(F.col("causas_externas_codigo_cie_10").isNotNull())
    .withColumn("anio",        F.col("anio").cast(IntegerType()))
    .withColumn("total",       F.col("total").cast(IntegerType()))
    .withColumn("hombres",     F.col("hombres").cast(IntegerType()))
    .withColumn("mujeres",     F.col("mujeres").cast(IntegerType()))
    # Bronze combina descripción + rango CIE en un campo: "Accidentes de transporte (V01-V99)"
    # → causa_de_muerte: texto descriptivo antes del paréntesis
    .withColumn(
        "causa_de_muerte",
        F.trim(F.regexp_replace(F.col("causas_externas_codigo_cie_10"), r'\s*\([^)]+\)\s*$', ''))
    )
    # → cie_10_norm: rango o código entre paréntesis (ej. V01-V99); sin paréntesis: normalize_cie10
    .withColumn(
        "cie_10_norm",
        F.when(
            F.col("causas_externas_codigo_cie_10").rlike(r'\('),
            F.regexp_extract(F.col("causas_externas_codigo_cie_10"), r'\(([^)]+)\)', 1)
        ).otherwise(normalize_cie10(F.col("causas_externas_codigo_cie_10")))
    )
    # Filtrar subtotal geográfico — esta tabla no tiene subtotal de causa
    .filter(F.upper(F.trim(F.col("departamento_de_ocurrencia"))) != F.lit("TODOS LOS DEPARTAMENTOS"))
    .dropDuplicates()
)

validate_cuadratura(df_silver, SILVER_TABLE)
df_silver = join_departamento(df_silver, "departamento_de_ocurrencia", ref_depto_df)
df_silver = add_silver_audit(df_silver, RUN_ID)

logger.info(f"[{SILVER_TABLE}] Silver: {df_silver.count():,} filas")
(
    df_silver.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_SCHEMA}.{SILVER_TABLE}")
)
logger.info(f"Escrito → {SILVER_SCHEMA}.{SILVER_TABLE}")

## Validación — Row counts, subtotales y resolución de departamentos

In [ ]:
_CHECKS = [
    (
        "ine_defunciones_sexo_depto_residencia_causas_muerte",
        "ine_defunciones_depto_residencia",
        "departamento_de_residencia",
    ),
    (
        "ine_defunciones_causas_externas_sexo_depto_ocurrencia",
        "ine_defunciones_causas_externas",
        "departamento_de_ocurrencia",
    ),
]

print(f"{'Tabla Silver':<38} {'Bronze':>8} {'Silver':>8} {'Filtradas':>10}")
print("-" * 68)
for bronze_name, silver_name, depto_col in _CHECKS:
    b_cnt = spark.table(f"{BRONZE_SCHEMA}.{bronze_name}").count()
    s_cnt = spark.table(f"{SILVER_SCHEMA}.{silver_name}").count()
    print(f"{silver_name:<38} {b_cnt:>8,} {s_cnt:>8,} {b_cnt - s_cnt:>10,}")

print("\nNota: 'Filtradas' = subtotales geográficos y de causa eliminados en Silver.")

print("\n── Verificar que no quedan subtotales en Silver ──")
for bronze_name, silver_name, depto_col in _CHECKS:
    remaining = (
        spark.table(f"{SILVER_SCHEMA}.{silver_name}")
        .filter(F.upper(F.trim(F.col(depto_col))) == "TODOS LOS DEPARTAMENTOS")
        .limit(10)
        .count()
    )
    status = "OK — sin subtotales" if remaining == 0 else f"WARN — {remaining} subtotales encontrados"
    print(f"  {silver_name}: {status}")

print("\n── Departamentos sin match en catálogo ──")
for bronze_name, silver_name, depto_col in _CHECKS:
    sin_match = (
        spark.table(f"{SILVER_SCHEMA}.{silver_name}")
        .filter(F.col("departamento_oficial").isNull())
        .select(depto_col).distinct()
    )
    cnt = sin_match.count()
    if cnt > 0:
        print(f"  {silver_name}: {cnt} valores sin match — revisar catálogo")
        sin_match.show(truncate=False)
    else:
        print(f"  {silver_name}: todos los departamentos resueltos OK")

print("\n── Muestra ine_defunciones_depto_residencia (limit 5) ──")
(
    spark.table(f"{SILVER_SCHEMA}.ine_defunciones_depto_residencia")
    .select("anio", "departamento_de_residencia", "departamento_oficial",
            "codigo_cie_10", "cie_10_norm", "total", "hombres", "mujeres")
    .limit(5)
    .show(truncate=False)
)

print("\n── Muestra ine_defunciones_causas_externas — campo Bronze separado en Silver (limit 5) ──")
(
    spark.table(f"{SILVER_SCHEMA}.ine_defunciones_causas_externas")
    .select("anio", "departamento_de_ocurrencia", "departamento_oficial",
            "causas_externas_codigo_cie_10", "causa_de_muerte", "cie_10_norm",
            "total", "hombres", "mujeres")
    .limit(5)
    .show(truncate=False)
)

print("\n── Valores únicos de cie_10_norm en causas_externas (deben ser rangos como V01-V99) ──")
(
    spark.table(f"{SILVER_SCHEMA}.ine_defunciones_causas_externas")
    .select("causas_externas_codigo_cie_10", "causa_de_muerte", "cie_10_norm")
    .dropDuplicates(["causas_externas_codigo_cie_10"])
    .orderBy("cie_10_norm")
    .show(30, truncate=False)
)